In [1]:
import math
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"
import inspect
from typing import List, Tuple, Dict, Any
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, load_from_disk
from transformers import (
    LEDForConditionalGeneration,
    LEDTokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import numpy as np
import re

/home/23ucs642/miniconda3/envs/myenv/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/23ucs642/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import PegasusTokenizer

# Change this to any Pegasus checkpoint you want to inspect
model_name = "google/pegasus-xsum"

tokenizer = PegasusTokenizer.from_pretrained(model_name)

print("Model:", model_name)
print("Model max length:", tokenizer.model_max_length)
print("Tokenizer max length:", tokenizer.max_len_single_sentence)
print("Max model input size:", tokenizer.max_model_input_sizes.get(model_name))


tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MauryaP\.cache\huggingface\hub\models--google--pegasus-xsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Model: google/pegasus-xsum
Model max length: 512
Tokenizer max length: 511


AttributeError: PegasusTokenizer has no attribute max_model_input_sizes

In [2]:
GLOBAL_UTT_BOUNDS = None 

class VanillaAdapter(nn.Module):
    """Down-up adapter: h' = h + W_up( f( LN( W_down(h) ) ) )"""
    def __init__(self, d_model: int, d_bottleneck: int = 256, non_linearity=F.relu):
        super().__init__()
        self.W_down = nn.Linear(d_model, d_bottleneck)
        self.layernorm = nn.LayerNorm(d_bottleneck)
        self.non_linearity = non_linearity
        self.W_up = nn.Linear(d_bottleneck, d_model)
        if self.W_up.bias is not None:
            nn.init.zeros_(self.W_up.bias)

    def forward(self, hidden_states: torch.Tensor):
        # hidden_states: (B, L, d_model)
        h_down = self.W_down(hidden_states)            # (B, L, d_b)
        h_down = self.layernorm(h_down)
        h_down = self.non_linearity(h_down)
        h_up = self.W_up(h_down)                       # (B, L, d_model)
        return hidden_states + h_up

In [3]:
class AttentionPooling(nn.Module):
    """Learnable pooling over tokens in a span."""
    def __init__(self, d_hidden: int, temperature: float = 1.0):
        super().__init__()
        self.scorer = nn.Linear(d_hidden, 1)
        nn.init.normal_(self.scorer.weight, std=0.02)
        if self.scorer.bias is not None:
            nn.init.zeros_(self.scorer.bias)
        self.temperature = float(temperature)

    def forward(self, tokens: torch.Tensor):
        # tokens: (T, d)
        if tokens.numel() == 0:
            return tokens.new_zeros(tokens.size(-1))
        scores = self.scorer(tokens).squeeze(-1)  # (T,)
        scores = scores / max(self.temperature, 1e-6)
        weights = F.softmax(scores, dim=0)       # (T,)
        pooled = (weights.unsqueeze(-1) * tokens).sum(dim=0)
        return pooled


In [4]:
class HierarchicalAdapterWithPooling(nn.Module):
    """
    Hierarchical adapter with mean or attention pooling.
    Operates at encoder layer: down-project tokens -> pool per utterance -> cross-utt attention ->
    broadcast back -> up-project and residual-add.
    """
    def __init__(
        self,
        d_model: int,
        d_bottleneck: int = 256,
        pooling: str = "attn",  # "attn" or "mean"
        tau: float = 0.1,
        use_proj_for_similarity: bool = True,
    ):
        super().__init__()
        assert pooling in ("attn", "mean")
        self.d_model = d_model
        self.d_bottleneck = d_bottleneck
        self.tau = tau
        self.pooling = pooling

        self.W_down = nn.Linear(d_model, d_bottleneck)
        self.layernorm = nn.LayerNorm(d_bottleneck)
        self.activation = F.relu

        if pooling == "attn":
            self.attn_pool = AttentionPooling(d_bottleneck, temperature=1.0)
        else:
            self.attn_pool = None

        if use_proj_for_similarity:
            self.W_sim_q = nn.Linear(d_bottleneck, d_bottleneck, bias=True)
            self.W_sim_k = nn.Linear(d_bottleneck, d_bottleneck, bias=True)
        else:
            self.W_sim_q = None
            self.W_sim_k = None

        self.W_sim_v = nn.Linear(d_bottleneck, d_bottleneck, bias=True)
        self.W_up = nn.Linear(d_bottleneck, d_model)
        if self.W_up.bias is not None:
            nn.init.zeros_(self.W_up.bias)

    def _pool_span(self, span_tokens: torch.Tensor):
        # span_tokens: (T, d_b)
        if span_tokens.size(0) == 0:
            return span_tokens.new_zeros(self.d_bottleneck)
        if self.pooling == "attn":
            return self.attn_pool(span_tokens)
        return span_tokens.mean(dim=0)

    def forward(self, hidden_states: torch.Tensor, utterance_boundaries: List[List[Tuple[int, int]]] = None):
        """
        hidden_states: (B, L, d_model)
        returns: hidden_states + up_projected_context
        """
        B, L, _ = hidden_states.shape
        device = hidden_states.device

        if utterance_boundaries is None:
            utterance_boundaries = [[(0, L)] for _ in range(B)]

        # 1. down project
        h_down = self.W_down(hidden_states)  # (B, L, d_b)
        h_down = self.layernorm(h_down)
        h_down = self.activation(h_down)

        # 2. build utterance vectors
        utt_vectors = []
        for b in range(B):
            spans = utterance_boundaries[b] if len(utterance_boundaries[b]) > 0 else [(0, L)]
            per_utt = []
            for (s, e) in spans:
                s_clamped = max(0, min(s, L))
                e_clamped = max(s_clamped, min(e, L))
                if e_clamped <= s_clamped:
                    per_utt.append(torch.zeros(self.d_bottleneck, device=device))
                else:
                    span_tokens = h_down[b, s_clamped:e_clamped]  # (T, d_b)
                    pooled = self._pool_span(span_tokens)         # (d_b,)
                    per_utt.append(pooled)
            utt_vectors.append(torch.stack(per_utt, dim=0))  # (U_b, d_b)

        # 3. pad to max_U for batch matmul
        max_U = max(uv.size(0) for uv in utt_vectors)
        padded_uv = []
        masks = []
        for uv in utt_vectors:
            U = uv.size(0)
            if U < max_U:
                pad = torch.zeros(max_U - U, self.d_bottleneck, device=device)
                padded = torch.cat([uv, pad], dim=0)
                mask = torch.tensor([1]*U + [0]*(max_U-U), device=device)
            else:
                padded = uv
                mask = torch.tensor([1]*U, device=device)
            padded_uv.append(padded)
            masks.append(mask)
        utterance_tensor = torch.stack(padded_uv, dim=0)  # (B, max_U, d_b)
        mask_tensor = torch.stack(masks, dim=0)           # (B, max_U)

        # 4. similarity / context
        Q = self.W_sim_q(utterance_tensor) if self.W_sim_q is not None else utterance_tensor
        K = self.W_sim_k(utterance_tensor) if self.W_sim_k is not None else utterance_tensor
        V = self.W_sim_v(utterance_tensor)

        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_bottleneck)  # (B, max_U, max_U)
        # mask padding positions in K
        dtype = scores.dtype
        min_val = torch.finfo(dtype).min  # safe min for fp16 or fp32
        scores = scores.masked_fill(mask_tensor.unsqueeze(1) == 0, min_val)
        attn = F.softmax(scores / max(self.tau, 1e-6), dim=-1)
        context = torch.matmul(attn, V)  # (B, max_U, d_b)

        # 5. scatter context back to token spans
        h_up_tokens = h_down.clone()
        for b in range(B):
            spans = utterance_boundaries[b]
            U_b = min(len(spans), context.size(1))
            for u in range(U_b):
                s, e = spans[u]
                s_clamped = max(0, min(s, L))
                e_clamped = max(s_clamped, min(e, L))
                if e_clamped <= s_clamped:
                    continue
                ctx = context[b, u]
                h_up_tokens[b, s_clamped:e_clamped] = h_up_tokens[b, s_clamped:e_clamped] + ctx.unsqueeze(0)

        # 6. up-project and residual add
        h_out = self.W_up(h_up_tokens)
        return hidden_states + h_out


In [5]:
def _locate_led_encoder_decoder(model: nn.Module):
    if hasattr(model, "led") and hasattr(model.led, "encoder"):
        return model.led.encoder, model.led.decoder
    if hasattr(model, "model") and hasattr(model.model, "led") and hasattr(model.model.led, "encoder"):
        return model.model.led.encoder, model.model.led.decoder
    if hasattr(model, "model") and hasattr(model.model, "encoder"):
        return model.model.encoder, model.model.decoder
    raise RuntimeError("Cannot locate LED encoder/decoder on provided model.")


In [6]:
def attach_hierarchical_and_vanilla_adapters_monkeypatch(
    model: nn.Module,
    d_bottleneck_enc: int = 256,
    d_bottleneck_dec: int = 256,
    pooling: str = "attn",
    tau: float = 0.1,
    use_proj_for_similarity: bool = True,
    enc_top_k: int = -1,  # attach hierarchical adapter to top-k encoder layers; -1 => all
):
    
    import inspect
    import re
    import types

    if hasattr(model, "led") and hasattr(model.led, "encoder"):
        encoder = model.led.encoder
        decoder = model.led.decoder
    elif hasattr(model, "model") and hasattr(model.model, "led") and hasattr(model.model.led, "encoder"):
        encoder = model.model.led.encoder
        decoder = model.model.led.decoder
    elif hasattr(model, "model") and hasattr(model.model, "encoder"):
        encoder = model.model.encoder
        decoder = model.model.decoder
    else:
        raise RuntimeError("Cannot locate LED encoder/decoder on provided model.")

    def _strip_unexpected_kw_from_errmsg(err_msg: str):
        names = re.findall(r"unexpected keyword argument '([^']+)'", str(err_msg))
        names += re.findall(r'unexpected keyword argument "([^"]+)"', str(err_msg))
        names += re.findall(r"got an unexpected keyword argument:? '([^']+)'", str(err_msg))
        return list(dict.fromkeys(names))

    def _get_accepted_param_names_for_callable(callable_obj):
        
        try:
            if hasattr(callable_obj, "__func__"):
                func = callable_obj.__func__
            else:
                func = callable_obj
            sig = inspect.signature(func)
            names = list(sig.parameters.keys())
            if names and names[0] == "self":
                names = names[1:]
            return set(names)
        except (ValueError, TypeError):
            return None

    def _derive_accepted_from_error_and_retry(bound_callable, args_tuple, kwargs, err_msg):
        unexpected = _strip_unexpected_kw_from_errmsg(err_msg)
        if not unexpected:
            accepted = set() 
        else:
            accepted = set(kwargs.keys()) - set(unexpected)
        filtered = {k: v for k, v in kwargs.items() if k in accepted}
        return bound_callable(*args_tuple, **filtered), accepted

    # Helper to create a uniform "call original forward" function that avoids double-binding.
    def _make_call_old_fn(original_forward, layer_obj):
       
        orig = original_forward
        is_bound_method = hasattr(orig, "__self__") and orig.__self__ is layer_obj

        if is_bound_method:
            def call_old(self, *args, **kwargs):
                return orig(*args, **kwargs)
        else:
            def call_old(self, *args, **kwargs):
                bound = orig.__get__(self, type(self))
                return bound(*args, **kwargs)

        return call_old

    # Encoder
    n_enc_layers = len(encoder.layers)
    if enc_top_k is None or enc_top_k < 0:
        enc_start = 0
    else:
        enc_start = max(0, n_enc_layers - enc_top_k)

    for i, layer in enumerate(encoder.layers):
        attach_adapter = (i >= enc_start)
        if attach_adapter:
            adapter = HierarchicalAdapterWithPooling(
                d_model=model.config.d_model,
                d_bottleneck=d_bottleneck_enc,
                pooling=pooling,
                tau=tau,
                use_proj_for_similarity=use_proj_for_similarity,
            )
            layer.hierarchical_adapter = adapter

        original_forward = layer.forward
        call_old = _make_call_old_fn(original_forward, layer)
        accepted = _get_accepted_param_names_for_callable(original_forward)
        layer._monkey_accepted = accepted 

        def make_new_forward_enc(call_old_fn, attach_flag):
            def new_forward_enc(self, hidden_states, *args, **kwargs):
                acc = getattr(self, "_monkey_accepted", None)
                if acc is not None:
                    if acc:
                        filtered = {k: v for k, v in kwargs.items() if k in acc}
                        outputs = call_old_fn(self, hidden_states, *args, **filtered)
                    else:
                        # empty set => call without kwargs
                        outputs = call_old_fn(self, hidden_states, *args)
                else:
                    try:
                        outputs = call_old_fn(self, hidden_states, *args, **kwargs)
                        self._monkey_accepted = set(kwargs.keys())
                    except TypeError as e:
                        try:
                            outputs, derived = _derive_accepted_from_error_and_retry(
                                call_old_fn.__get__(self, type(self)) if hasattr(call_old_fn, "__get__") else call_old_fn,
                                (self, hidden_states) if False else (hidden_states,) + args,
                                kwargs,
                                str(e),
                            )
                            self._monkey_accepted = set(derived)
                        except Exception:
                            # last resort: call without kwargs
                            outputs = call_old_fn(self, hidden_states, *args)
                            self._monkey_accepted = set()

                hidden = outputs[0] if isinstance(outputs, tuple) else outputs

                if not attach_flag:
                    return outputs

                ub = GLOBAL_UTT_BOUNDS
                if ub is None:
                    B, L, _ = hidden.shape
                    ub_local = [[(0, L)] for _ in range(B)]
                else:
                    ub_local = ub

                new_hidden = self.hierarchical_adapter(hidden, ub_local)
                if isinstance(outputs, tuple):
                    return (new_hidden,) + outputs[1:]
                return new_hidden

            return new_forward_enc

        layer.forward = make_new_forward_enc(call_old, attach_adapter).__get__(layer, type(layer))

    # Decoder
    for layer in decoder.layers:
        adapter = VanillaAdapter(d_model=model.config.d_model, d_bottleneck=d_bottleneck_dec)
        layer.vanilla_adapter = adapter

        original_forward = layer.forward
        call_old = _make_call_old_fn(original_forward, layer)
        accepted = _get_accepted_param_names_for_callable(original_forward)
        layer._monkey_accepted = accepted

        def make_new_forward_dec(call_old_fn):
            def new_forward_dec(self, hidden_states, *args, **kwargs):
                acc = getattr(self, "_monkey_accepted", None)
                if acc is not None:
                    if acc:
                        filtered = {k: v for k, v in kwargs.items() if k in acc}
                        outputs = call_old_fn(self, hidden_states, *args, **filtered)
                    else:
                        outputs = call_old_fn(self, hidden_states, *args)
                else:
                    try:
                        outputs = call_old_fn(self, hidden_states, *args, **kwargs)
                        self._monkey_accepted = set(kwargs.keys())
                    except TypeError as e:
                        try:
                            outputs, derived = _derive_accepted_from_error_and_retry(
                                call_old_fn.__get__(self, type(self)) if hasattr(call_old_fn, "__get__") else call_old_fn,
                                (hidden_states,) + args,
                                kwargs,
                                str(e),
                            )
                            self._monkey_accepted = set(derived)
                        except Exception:
                            outputs = call_old_fn(self, hidden_states, *args)
                            self._monkey_accepted = set()

                hidden = outputs[0] if isinstance(outputs, tuple) else outputs
                new_hidden = self.vanilla_adapter(hidden)
                if isinstance(outputs, tuple):
                    return (new_hidden,) + outputs[1:]
                return new_hidden

            return new_forward_dec

        layer.forward = make_new_forward_dec(call_old).__get__(layer, type(layer))

    print(
        "monkeypatch complete. "
        f"(hierarchical adapters attached to encoder top-k layers = {n_enc_layers - enc_start if enc_top_k and enc_top_k > 0 else 'all'})"
    )


In [7]:
def freeze_base_model_except_adapters(model: nn.Module):
    """Freeze all parameters except those whose name contains 'hierarchical_adapter' or 'vanilla_adapter'"""
    for n, p in model.named_parameters():
        if ("hierarchical_adapter" in n) or ("vanilla_adapter" in n):
            p.requires_grad = True
        else:
            p.requires_grad = False


def count_trainable_params(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

In [8]:
def build_utterance_boundaries_from_segments(segments: List[str], tokenizer: LEDTokenizerFast, max_length: int = 8192):
    """
    Tokenizes each segment without special tokens and returns:
      - flattened input_ids (list[int]) truncated to max_length
      - utterance_boundaries: list of (start,end) indices relative to flattened list
    """
    tokenized = [tokenizer(s, add_special_tokens=False)["input_ids"] for s in segments]
    input_ids = []
    boundaries: List[Tuple[int, int]] = []
    start = 0
    for ids in tokenized:
        if len(ids) == 0:
            continue
        end = start + len(ids)
        if end > max_length:
            remaining = max_length - start
            if remaining <= 0:
                break
            ids = ids[:remaining]
            end = start + len(ids)
            input_ids.extend(ids)
            boundaries.append((start, end))
            break
        input_ids.extend(ids)
        boundaries.append((start, end))
        start = end
    return input_ids, boundaries

In [9]:
def tokenize_example_with_segments(example: Dict[str, Any], tokenizer: LEDTokenizerFast, max_input: int = 8192, max_target: int = 516):
    """
    Returns dict with 'input_ids' (list[int]), 'labels' (list[int]), 'utterance_boundaries' (list of spans)
    """
    segments = []
    if "transcript_segments" in example and isinstance(example["transcript_segments"], dict):
        segments = example["transcript_segments"].get("text", [])
    if not segments and "transcript" in example:
        segments = [example["transcript"]]

    input_ids, boundaries = build_utterance_boundaries_from_segments(segments, tokenizer, max_length=max_input)
    if len(input_ids) == 0 and "transcript" in example:
        enc = tokenizer(example["transcript"], add_special_tokens=False, truncation=True, max_length=max_input)
        input_ids = enc["input_ids"]
        boundaries = [(0, len(input_ids))] if len(input_ids) > 0 else []

    target_text = example.get("summary") or example.get("abstract") or ""
    label_enc = tokenizer(target_text, truncation=True, max_length=max_target, padding=False)
    labels = label_enc["input_ids"]
    return {"input_ids": input_ids, "labels": labels, "utterance_boundaries": boundaries}

In [10]:
class CollatorWithUtteranceBoundaries:
    def __init__(self, tokenizer: LEDTokenizerFast, pad_to_multiple_of: int = None):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, batch: List[Dict[str, Any]]):
        """
        batch: list of dicts from dataset with keys 'input_ids', 'labels', 'utterance_boundaries'
        Returns padded tensors dict acceptable by Trainer.
        """
        global GLOBAL_UTT_BOUNDS
        ub_batch = [ex.get("utterance_boundaries", [(0, len(ex.get("input_ids", [])))]) for ex in batch]
        GLOBAL_UTT_BOUNDS = ub_batch
        input_ids_list = [ex.get("input_ids", []) for ex in batch]
        labels_list = [ex.get("labels", []) for ex in batch]

        max_len = max((len(x) for x in input_ids_list), default=0)
        if self.pad_to_multiple_of is not None and max_len % self.pad_to_multiple_of != 0:
            max_len = ((max_len // self.pad_to_multiple_of) + 1) * self.pad_to_multiple_of
        padded_inputs = [ids + [self.tokenizer.pad_token_id] * (max_len - len(ids)) for ids in input_ids_list]
        attention_mask = [[1] * len(ids) + [0] * (max_len - len(ids)) for ids in input_ids_list]

        # pad labels with -100 for ignore index
        max_label_len = max((len(l) for l in labels_list), default=0)
        padded_labels = [l + [-100] * (max_label_len - len(l)) for l in labels_list]

        batch_output = {
            "input_ids": torch.tensor(padded_inputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long),
        }
        return batch_output

In [11]:
def tokenize_example_with_segments(example):
    # segments text stored in example["transcript_segments"]["text"]
    segments = example.get("transcript_segments", {}).get("text", [])
    input_ids, boundaries = build_utterance_boundaries_from_segments(segments, tokenizer, max_length=max_input)

    if len(input_ids) == 0:
        enc = tokenizer(example["transcript"], add_special_tokens=False, truncation=True, max_length=max_input)
        input_ids = enc["input_ids"]
        boundaries = [(0, len(input_ids))] if len(input_ids) > 0 else []

    label_enc = tokenizer(example.get("summary", example.get("abstract", "")),
                          truncation=True, max_length=max_target, padding=False)
    labels = label_enc["input_ids"]

    return {
        "input_ids": input_ids,
        "labels": labels,
        "utterance_boundaries": boundaries
    }

# Example usage:
# dataset = load_dataset("gigant/tib")
# tokenized = dataset.map(tokenize_example_with_segments, remove_columns=dataset["train"].column_names)
# tokenized.save_to_disk("./tib_tokenized_with_segments")


In [12]:
model_name = "allenai/led-base-16384"
cache_dir = None
out_dir = "/home/23ucs642/Video_Summarization_Translation/led_hier_adapter_run"
max_input = 8192
max_target = 516
bottleneck_enc = 256
bottleneck_dec = 128
pooling = "attn" 
tau = 0.1
batch_size = 1
num_train_epochs = 10
gradient_accumulation_steps = 4
learning_rate = 2e-4
fp16 = True
pad_to_multiple_of = 8
num_proc = 4 

# load tokenizer + dataset
tokenizer = LEDTokenizerFast.from_pretrained(model_name)
# ensure pad_token exists
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})

raw = load_from_disk("/home/23ucs642/Video_Summarization_Translation/tib_dataset")

In [13]:
# # tokenize and build boundaries
# def map_fn(ex):
#     return tokenize_example_with_segments(ex, tokenizer, max_input=max_input, max_target=max_target)

# tokenized = raw.map(
#     tokenize_example_with_segments,
#     fn_kwargs={},
#     remove_columns=raw["train"].column_names,
#     num_proc=num_proc,
# )
# tokenized.save_to_disk(os.path.join(out_dir, "tokenized"))  # optional

# train_ds = tokenized["train"]
# val_ds = tokenized["validation"] if "validation" in tokenized else tokenized["test"]

# # Important: return python dicts to collator
# train_ds.set_format(type="python")
# val_ds.set_format(type="python")

In [14]:
tokenized = load_from_disk(os.path.join(out_dir, "tokenized"))
train_ds = tokenized["train"]
val_ds = tokenized["validation"] if "validation" in tokenized else tokenized["test"]
train_ds.set_format(type="python")
val_ds.set_format(type="python")

In [15]:
model = LEDForConditionalGeneration.from_pretrained(model_name)
attach_hierarchical_and_vanilla_adapters_monkeypatch( model, d_bottleneck_enc=bottleneck_enc, d_bottleneck_dec=bottleneck_dec, pooling=pooling, tau=tau, ) # freeze base model params 
freeze_base_model_except_adapters(model) 
trainable, total = count_trainable_params(model) 
print(f"Trainable params: {trainable} / {total} ({100.*trainable/total:.4f}%)")

monkeypatch complete. (hierarchical adapters attached to encoder top-k layers = all)
Trainable params: 4740870 / 166585350 (2.8459%)


In [16]:
for name, p in model.named_parameters():
    if "adapter" in name and "W_down.weight" in name:
        print(name, p.data.mean().item(), p.data.std().item())


led.encoder.layers.0.hierarchical_adapter.W_down.weight 3.190268762409687e-05 0.020800331607460976
led.encoder.layers.1.hierarchical_adapter.W_down.weight 2.7157315344084054e-05 0.02080359496176243
led.encoder.layers.2.hierarchical_adapter.W_down.weight -2.824063994921744e-05 0.020847853273153305
led.encoder.layers.3.hierarchical_adapter.W_down.weight 0.00010028830729424953 0.02082366868853569
led.encoder.layers.4.hierarchical_adapter.W_down.weight 6.26891414867714e-05 0.020840024575591087
led.encoder.layers.5.hierarchical_adapter.W_down.weight -6.688886787742376e-05 0.020834479480981827
led.decoder.layers.0.vanilla_adapter.W_down.weight -3.4626104024937376e-05 0.02082928828895092
led.decoder.layers.1.vanilla_adapter.W_down.weight 5.178657011128962e-05 0.02079293504357338
led.decoder.layers.2.vanilla_adapter.W_down.weight -2.7665519155561924e-05 0.020855287089943886
led.decoder.layers.3.vanilla_adapter.W_down.weight 8.67794660734944e-05 0.02086845226585865
led.decoder.layers.4.vanilla_

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=6e-4,
    num_train_epochs=10,
    fp16=fp16,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=100,
    remove_unused_columns=False,   
    dataloader_num_workers=4,   
    gradient_checkpointing=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", 
    greater_is_better=False,
    save_total_limit = 2,
)

data_collator = CollatorWithUtteranceBoundaries(tokenizer, pad_to_multiple_of=model.config.attention_window[0])

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# --- train ---
trainer.train()
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
print("Training finished. Model + tokenizer saved to:", out_dir)

/tmp/ipykernel_57416/3376533384.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss
100,8.171300,7.785555
200,7.878900,7.785555
300,7.861800,7.785555
400,7.840300,7.785555
500,7.855700,7.785555


In [18]:
for name, p in model.named_parameters():
    if "adapter" in name and "W_down.weight" in name:
        print(name, p.data.mean().item(), p.data.std().item())


led.encoder.layers.0.hierarchical_adapter.W_down.weight -8.976904064184055e-05 0.020973069593310356
led.encoder.layers.1.hierarchical_adapter.W_down.weight 4.069688202434918e-06 0.020939525216817856
led.encoder.layers.2.hierarchical_adapter.W_down.weight -8.801609510555863e-05 0.021003322675824165
led.encoder.layers.3.hierarchical_adapter.W_down.weight -5.0679416744969785e-05 0.020928703248500824
led.encoder.layers.4.hierarchical_adapter.W_down.weight -5.553782102651894e-05 0.020932795479893684
led.encoder.layers.5.hierarchical_adapter.W_down.weight -9.170849807560444e-06 0.02098727598786354
led.decoder.layers.0.vanilla_adapter.W_down.weight -3.2508716685697436e-05 0.02092933841049671
led.decoder.layers.1.vanilla_adapter.W_down.weight -3.908643520844635e-06 0.020969368517398834
led.decoder.layers.2.vanilla_adapter.W_down.weight 9.117651643464342e-06 0.0210038460791111
led.decoder.layers.3.vanilla_adapter.W_down.weight -4.521629307419062e-06 0.020915133878588676
led.decoder.layers.4.van

In [31]:
all_predictions, all_references = [], []

for inputs, refs, ub_batch in tqdm(loader, desc="Evaluating in batches"):
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # Set utterance boundaries globally (so hierarchical adapters can use them)
    GLOBAL_UB = ub_batch

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET,
            num_beams=2,
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=2.0,
            encoder_no_repeat_ngram_size=3,
            min_length=50,
        )

    preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    all_predictions.extend(preds)
    all_references.extend(refs)

Evaluating in batches: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.30s/it]


In [32]:
GLOBAL_UB = None

# --- ROUGE evaluation ---
rouge = evaluate.load("rouge")
print("\nComputing ROUGE scores...")
results = rouge.compute(predictions=all_predictions, references=all_references, use_stemmer=True)
results = {k: round(v * 100, 2) for k, v in results.items()}

print("\n==== ROUGE Evaluation Results ====")
for k, v in results.items():
    print(f"{k}: {v}")


Computing ROUGE scores...

==== ROUGE Evaluation Results ====
rouge1: 17.35
rouge2: 4.12
rougeL: 13.27
rougeLsum: 13.27


In [33]:
all_references[0], all_predictions[0]

('Professor de Cooman takes the topic of rod and bar manufacture, including non-destructive testing. Both the production and metallurgy of the alloys is described. This is a part of a course of lectures given at the Graduate Institute of Ferrous Technology, POSTECH, Republic of Korea.',
 'In this article, we will discuss how to improve the performance of a computer system. We will also cover the basics of computing in general relativity and quantum mechanics. The first section of our talk will focus on the topic of computational complexity. This section will focus specifically on the importance of information security. It will cover the following topics: How to build an efficient computer system for solving complex mathematical problems. What can be done?What can be achieved by improving the efficiency of computers? In this section, we look at some key features of the computer system that make it possible to solve complex numerical problems. For example, there was a problem with the wa